In [ ]:
# Import các thư viện cần thiết
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssdlite import SSDLiteHead # Nhập SSDLiteHead từ module ssdlite
from torchvision.transforms import functional as F
import os
import cv2
import numpy as np
import xml.etree.ElementTree as ET
import random 

# Phát hiện cháy rừng từ ảnh vệ tinh bằng SSD Lite

Notebook này sử dụng mô hình **SSD Lite với MobileNet V3** để phát hiện các đám cháy rừng trong ảnh vệ tinh.

## Import thư viện

Các thư viện chính được sử dụng:
- **PyTorch**: Framework deep learning
- **Torchvision**: Mô hình SSD Lite và các tiện ích
- **OpenCV**: Xử lý ảnh
- **Albumentations**: Data augmentation
- **XML**: Đọc annotations từ file XML

---

##  CẤU HÌNH CƠ BẢN 

In [ ]:

DATA_DIR = os.environ.get('FIRE_VOC_ROOT', 'data/processed/voc')

# Kiểm tra sự tồn tại của thư mục 'train' để đảm bảo cấu trúc dữ liệu đúng
if not os.path.exists(os.path.join(DATA_DIR, 'train')):
    print(f"LỖI: Không tìm thấy thư mục 'train' trong {DATA_DIR}!")
else:
    print(f"Đã tìm thấy thư mục dữ liệu tại: {DATA_DIR}")

SAVE_PATH = os.environ.get('FIRE_SSDLITE_OUTPUT', 'artifacts/checkpoints/ssdlite')
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"Mô hình sẽ được lưu tại: {SAVE_PATH}")

---

## Các hàm đánh giá mô hình

Để đánh giá hiệu suất của mô hình phát hiện đối tượng, chúng ta sử dụng các metric:
- **IoU (Intersection over Union)**: Đo độ chồng chéo giữa bounding box dự đoán và ground truth
- **AP (Average Precision)**: Độ chính xác trung bình
- **mAP@0.5**: Mean Average Precision với ngưỡng IoU = 0.5

### 1. Hàm tính IoU


In [ ]:
def compute_iou(box1, box2):
    """
    box: [xmin, ymin, xmax, ymax]
    """
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union = area1 + area2 - inter_area

    if union == 0:
        return 0.0

    return inter_area / union


### 2. Hàm tính Average Precision (AP)

Hàm này tính toán AP từ các giá trị recall và precision đã được tính toán trước đó.


In [ ]:
import numpy as np

def compute_ap(recall, precision):
    recall = np.concatenate(([0.0], recall, [1.0]))
    precision = np.concatenate(([0.0], precision, [0.0]))

    for i in range(len(precision)-1, 0, -1):
        precision[i-1] = max(precision[i-1], precision[i])

    indices = np.where(recall[1:] != recall[:-1])[0]
    ap = np.sum((recall[indices + 1] - recall[indices]) * precision[indices + 1])
    return ap


### 3. Hàm đánh giá mAP@0.5

Hàm này đánh giá mô hình trên tập validation/test bằng cách:
1. Dự đoán các bounding boxes trên tất cả ảnh
2. So khớp với ground truth boxes sử dụng IoU threshold = 0.5
3. Tính toán precision và recall
4. Tính AP từ precision-recall curve


In [ ]:
def evaluate_map_50(model, dataloader, device, score_thresh=0.05):
    model.eval()

    detections = []   # (score, box, img_id)
    gt_boxes_all = {} # img_id -> gt_boxes
    gt_used = {}      # img_id -> used flags

    img_id = 0

    with torch.no_grad():
        for images, targets in dataloader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                gt_boxes = target['boxes'].cpu().numpy()

                gt_boxes_all[img_id] = gt_boxes
                gt_used[img_id] = np.zeros(len(gt_boxes))

                pred_boxes = output['boxes'].cpu().numpy()
                scores = output['scores'].cpu().numpy()
                labels = output['labels'].cpu().numpy()

                for box, score, label in zip(pred_boxes, scores, labels):
                    if score < score_thresh:
                        continue
                    if label != 1:  # fire
                        continue

                    detections.append((score, box, img_id))

                img_id += 1

    total_gt = sum(len(v) for v in gt_boxes_all.values())
    if total_gt == 0 or len(detections) == 0:
        return 0.0

    # Sort theo confidence
    detections.sort(key=lambda x: x[0], reverse=True)

    TP = np.zeros(len(detections))
    FP = np.zeros(len(detections))

    for i, (score, box, img_id) in enumerate(detections):
        gt_boxes = gt_boxes_all[img_id]
        used = gt_used[img_id]

        if len(gt_boxes) == 0:
            FP[i] = 1
            continue

        ious = [compute_iou(box, gt) for gt in gt_boxes]
        best_iou = max(ious)
        best_idx = np.argmax(ious)

        if best_iou >= 0.5 and used[best_idx] == 0:
            TP[i] = 1
            used[best_idx] = 1
        else:
            FP[i] = 1

    TP = np.cumsum(TP)
    FP = np.cumsum(FP)

    recall = TP / (total_gt + 1e-6)
    precision = TP / (TP + FP + 1e-6)

    return compute_ap(recall, precision)


---

##  Data Augmentation và Transform

Sử dụng **Albumentations** để áp dụng các phép biến đổi ảnh:
- **Training**: Horizontal/Vertical flip, xoay 90 độ, điều chỉnh màu sắc
- **Validation**: Chỉ resize và normalize

Các phép biến đổi này giúp tăng tính đa dạng của dữ liệu và cải thiện khả năng generalization của mô hình.


## ĐỊNH NGHĨA BIẾN HUẤN LUYỆN 

---

## Dataset Class

Class `WildfireDataset` kế thừa từ `torch.utils.data.Dataset` để:
- Đọc ảnh từ thư mục
- Đọc annotations từ file XML (Pascal VOC format)
- Áp dụng transforms/augmentations
- Trả về ảnh và targets (boxes, labels) theo định dạng mà PyTorch detection models yêu cầu


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

NUM_CLASSES = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH_SIZE = 8
MAX_EPOCHS = 200
PATIENCE = 20

LEARNING_RATE = 1e-3     
IMAGE_SIZE = 320

best_map = -1.0
epochs_no_improve = 0

CLASS_NAMES = ['__background__', 'fire']


---

##  Mô hình và hàm huấn luyện

### 1. Collate Function
Hàm này được sử dụng bởi DataLoader để gom các samples thành batch.

### 2. Khởi tạo mô hình SSD Lite
- Sử dụng pre-trained weights từ COCO dataset
- Chỉnh sửa số lượng classes để phù hợp với bài toán (2 classes: background + fire)

### 3. Hàm huấn luyện một epoch
Thực hiện forward pass, tính loss, backward pass và cập nhật weights.


In [ ]:
import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2

def get_transform(is_train):
    bbox_params = A.BboxParams(
    format='pascal_voc',
    label_fields=['class_labels'],
    clip=True,
    min_area=4,                  
    check_each_transform=False
)

    if is_train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ColorJitter(0.2, 0.2, 0.2, 0.01, p=0.5),

            A.Resize(IMAGE_SIZE, IMAGE_SIZE),
            A.ToFloat(max_value=255.0),
            ToTensorV2()
        ], bbox_params=bbox_params)
    else:
        return A.Compose([
            A.Resize(IMAGE_SIZE, IMAGE_SIZE),
            A.ToFloat(max_value=255.0),
            ToTensorV2()
        ], bbox_params=bbox_params)


---

##  Hàm main - Quá trình huấn luyện

Hàm này thực hiện toàn bộ quy trình huấn luyện:
1. **Khởi tạo datasets và dataloaders** cho train và validation
2. **Khởi tạo mô hình** SSD Lite với pre-trained weights
3. **Thiết lập optimizer** (SGD với momentum) và **learning rate scheduler** (MultiStepLR)
4. **Vòng lặp huấn luyện**:
   - Train một epoch
   - Đánh giá trên validation set (mAP@0.5)
   - Lưu model tốt nhất dựa trên mAP
   - Early stopping nếu không cải thiện sau PATIENCE epochs


In [ ]:
class WildfireDataset(Dataset):
    def __init__(self, root_dir, split='train'):
        self.root_dir = root_dir
        self.split_dir = os.path.join(root_dir, split)
        self.image_dir = self.split_dir
        self.annotation_dir = self.split_dir

        self.transforms = get_transform(is_train=(split == 'train'))

        self.image_files = [
            f for f in os.listdir(self.image_dir)
            if f.endswith('.jpg') and
            os.path.exists(os.path.join(self.annotation_dir, f.replace('.jpg', '.xml')))
        ]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        xml_path = os.path.join(self.annotation_dir, img_name.replace('.jpg', '.xml'))

        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f"Không đọc được ảnh: {img_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        boxes = []
        labels = []

        tree = ET.parse(xml_path)
        root = tree.getroot()

        for obj in root.findall('object'):
            class_name = obj.find('name').text.strip()
            if class_name != 'fire':
                continue

            label = 1
            bndbox = obj.find('bndbox')

            xmin = float(bndbox.find('xmin').text)
            ymin = float(bndbox.find('ymin').text)
            xmax = float(bndbox.find('xmax').text)
            ymax = float(bndbox.find('ymax').text)

            if xmax <= xmin or ymax <= ymin:
                continue
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(label)
        # Albumentations
        if len(boxes) == 0:
            transformed = self.transforms(
                image=image,
                bboxes=[],
                class_labels=[]
            )
            image = transformed['image']
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            transformed = self.transforms(
                image=image,
                bboxes=boxes,
                class_labels=labels
            )
            image = transformed['image']
            boxes = torch.tensor(transformed['bboxes'], dtype=torch.float32)
            labels = torch.tensor(transformed['class_labels'], dtype=torch.int64)

        target = {
            "boxes": boxes,    
            "labels": labels
        }
    return image, target


---

##  Chạy chương trình

Chạy hàm `main()` để bắt đầu quá trình huấn luyện mô hình.


In [ ]:

def collate_fn(batch):
    return tuple(zip(*batch))


import torch.nn as nn 
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models import MobileNet_V3_Large_Weights

def get_ssdlite_model(num_classes):
    model = ssdlite320_mobilenet_v3_large(
        weights=None,
        weights_backbone=MobileNet_V3_Large_Weights.IMAGENET1K_V1,
        num_classes=num_classes,
    )
    return model


def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    total_loss = 0
    
    from tqdm.auto import tqdm
    pbar = tqdm(data_loader, desc=f"Epoch {epoch}")

    for images, targets in pbar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
        
        pbar.set_postfix({"Loss": losses.item()})

    avg_loss = total_loss / len(data_loader)
    print(f"Epoch {epoch} hoàn thành. Mất mát trung bình: {avg_loss:.4f}")
    return avg_loss



In [ ]:

def main():
    print(f"Thiết bị sử dụng: {DEVICE}")

    train_dataset = WildfireDataset(root_dir=DATA_DIR, split='train')
    valid_dataset = WildfireDataset(root_dir=DATA_DIR, split='valid')
    test_dataset = WildfireDataset(root_dir=DATA_DIR, split='test')

    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=2, 
        collate_fn=collate_fn
    )

    valid_loader = DataLoader(
        valid_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=2, 
        collate_fn=collate_fn
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        collate_fn=collate_fn
    )
    
    print(f"Tổng số ảnh train: {len(train_dataset)}")
    print(f"Tổng số ảnh valid: {len(valid_dataset)}")
    print(f"Tổng số ảnh test: {len(test_dataset)}")

    model = get_ssdlite_model(NUM_CLASSES)
    model.to(DEVICE)

    params = [p for p in model.parameters() if p.requires_grad]
    
    optimizer = torch.optim.SGD(
    params,
    lr=1e-3,        
    momentum=0.9,
    weight_decay=5e-4
    )
    
    lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[80, 140, 180],
    gamma=0.1
    )

    best_loss = float('inf')

    best_map = -1.0
    epochs_no_improve = 0
    
    for epoch in range(1, MAX_EPOCHS + 1):
    
        print(f"\nEpoch {epoch}/{MAX_EPOCHS}")
    
        # ================= TRAIN =================
        train_loss = train_one_epoch(
            model, optimizer, train_loader, DEVICE, epoch
        )
    
        # ================= VALID =================
        print("Đánh giá mAP@0.5 trên tập validation...")
        val_map = evaluate_map_50(
            model, valid_loader, DEVICE
        )
    
        print(f"Epoch {epoch} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val mAP@0.5: {val_map:.4f}")

        lr_scheduler.step()
    

        if val_map > best_map:
            best_map = val_map
            epochs_no_improve = 0
    
            torch.save(
                model.state_dict(),
                os.path.join(SAVE_PATH, "best_ssd_model.pth")
            )
            print(f" mAP cải thiện ({best_map:.4f}) → Lưu model tốt nhất")
    
        else:
            epochs_no_improve += 1
            print(f"Không cải thiện mAP ({epochs_no_improve}/{PATIENCE})")
    
        # ================= EARLY STOPPING =================
        if epochs_no_improve >= PATIENCE:
            print("Model đã HỘI TỤ → DỪNG TRAIN")
            break
    
    
    best_checkpoint = os.path.join(SAVE_PATH, "best_ssd_model.pth")
    if not os.path.isfile(best_checkpoint):
        raise FileNotFoundError(best_checkpoint)
    model.load_state_dict(torch.load(best_checkpoint, map_location=DEVICE, weights_only=True))
    test_map = evaluate_map_50(model, test_loader, DEVICE)

    print("\n====================================")
    print("QUÁ TRÌNH HUẤN LUYỆN HOÀN TẤT")
    print(f"Best validation mAP@0.5: {best_map:.4f}")
    print(f"Test mAP@0.5: {test_map:.4f}")
    print("====================================")



In [ ]:

# --- 5. CHẠY MAIN ---
if __name__ == "__main__":
    try:
        from tqdm.auto import tqdm
    except ImportError:
        print("Cảnh báo: Thư viện 'tqdm' chưa được cài đặt. Tiến trình sẽ không có thanh hiển thị.")
        
    main()